# Pharmacy locations

**Section 1 — data prep for access analysis**

Discover pharmacies in and around **Caledonia**, **Orleans**, **Lamoille**, and **Washington** counties using the **Google Places API (New)** Nearby Search. We tile overlapping 30 km-radius search circles across the region plus a ~20-mile buffer. Results are deduplicated by `place_id` and cached to `data/pharmacies.json`.

We then preview search coverage and locations on a map, drop permanently closed sites, and write **`data/pharmacies_active.json`** for `01-pharmacy_access.ipynb` (clustering and drive times).

To refresh after a closure: delete `pharmacies.json` and re-run this notebook (or re-run to rebuild the active list from an updated cache).


In [1]:
import json
import math
import os
import time
from pathlib import Path

import folium
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

GOOGLE_API_KEY = os.environ.get("GOOGLE_ROUTES_API_KEY", "")
assert GOOGLE_API_KEY, "Set GOOGLE_ROUTES_API_KEY in .env (Places + Routes can share one key if both APIs are enabled)."
print(f"API key loaded: {GOOGLE_API_KEY[:8]}...")


API key loaded: AIzaSyAs...


In [2]:
PHARMACY_CACHE = DATA_DIR / "pharmacies.json"
PLACES_URL = "https://places.googleapis.com/v1/places:searchNearby"

# Bounding box covering the 4 target counties + ~20-mile buffer,
# reaching into neighboring VT counties and across the NH border.
SEARCH_BBOX = {"lat_min": 43.76, "lat_max": 45.30, "lon_min": -73.30, "lon_max": -71.26}
SEARCH_RADIUS_M = 30_000  # 30 km per circle


def search_nearby_pharmacies(center_lat, center_lon, radius_m, api_key):
    body = {
        "includedTypes": ["pharmacy"],
        "locationRestriction": {
            "circle": {
                "center": {"latitude": center_lat, "longitude": center_lon},
                "radius": radius_m,
            }
        },
        "maxResultCount": 20,
    }
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": (
            "places.displayName,places.formattedAddress,"
            "places.location,places.id,places.businessStatus"
        ),
    }
    resp = requests.post(PLACES_URL, json=body, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json().get("places", [])


def fetch_all_pharmacies(bbox, radius_m, api_key):
    """Tile overlapping search circles across bbox, deduplicate by place_id."""
    mid_lat = (bbox["lat_min"] + bbox["lat_max"]) / 2
    # ~60% of diameter for overlap between circles
    lat_step = radius_m * 1.2 / 111_000
    lon_step = lat_step / math.cos(math.radians(mid_lat))

    lats = np.arange(bbox["lat_min"], bbox["lat_max"] + lat_step * 0.1, lat_step)
    lons = np.arange(bbox["lon_min"], bbox["lon_max"] + lon_step * 0.1, lon_step)
    print(f"  Search grid: {len(lats)} × {len(lons)} = {len(lats) * len(lons)} circles "
          f"(radius {radius_m / 1000:.0f} km)")

    seen = {}
    n_calls = 0
    for lat in lats:
        for lon in lons:
            places = search_nearby_pharmacies(lat, lon, radius_m, api_key)
            n_calls += 1
            for p in places:
                pid = p["id"]
                if pid not in seen:
                    addr = p.get("formattedAddress", "")
                    parts = [s.strip() for s in addr.split(",")]
                    city = parts[1] if len(parts) >= 3 else ""
                    seen[pid] = {
                        "place_id": pid,
                        "name": p.get("displayName", {}).get("text", ""),
                        "address": addr,
                        "city": city,
                        "lat": p["location"]["latitude"],
                        "lon": p["location"]["longitude"],
                        "business_status": p.get("businessStatus", "OPERATIONAL"),
                    }
            time.sleep(0.1)

    print(f"  {n_calls} API calls → {len(seen)} unique pharmacies")
    return list(seen.values())


if PHARMACY_CACHE.exists():
    all_pharmacies = json.loads(PHARMACY_CACHE.read_text())
    print(f"Loaded {len(all_pharmacies)} pharmacies from cache ({PHARMACY_CACHE.name})")
else:
    print("Searching for pharmacies via Google Places API (Nearby Search)...")
    all_pharmacies = fetch_all_pharmacies(SEARCH_BBOX, SEARCH_RADIUS_M, GOOGLE_API_KEY)
    PHARMACY_CACHE.write_text(json.dumps(all_pharmacies, indent=2))
    print(f"Cached to {PHARMACY_CACHE.name}")


Loaded 224 pharmacies from cache (pharmacies.json)


In [3]:
mid_lat = (SEARCH_BBOX["lat_min"] + SEARCH_BBOX["lat_max"]) / 2
mid_lon = (SEARCH_BBOX["lon_min"] + SEARCH_BBOX["lon_max"]) / 2
lat_step = SEARCH_RADIUS_M * 1.2 / 111_000
lon_step = lat_step / math.cos(math.radians(mid_lat))

lats = np.arange(SEARCH_BBOX["lat_min"], SEARCH_BBOX["lat_max"] + lat_step * 0.1, lat_step)
lons = np.arange(SEARCH_BBOX["lon_min"], SEARCH_BBOX["lon_max"] + lon_step * 0.1, lon_step)

m = folium.Map(location=[mid_lat, mid_lon], zoom_start=8, tiles="cartodbpositron")

for lat in lats:
    for lon in lons:
        folium.Circle(
            location=[lat, lon],
            radius=SEARCH_RADIUS_M,
            color="#3388ff",
            weight=1,
            fill=True,
            fill_opacity=0.05,
        ).add_to(m)

for p in all_pharmacies:
    color = "red" if p.get("business_status") == "CLOSED_PERMANENTLY" else "green"
    folium.CircleMarker(
        location=[p["lat"], p["lon"]],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{p['name']} — {p['address']}",
    ).add_to(m)

# Draw bounding box
bbox_coords = [
    [SEARCH_BBOX["lat_min"], SEARCH_BBOX["lon_min"]],
    [SEARCH_BBOX["lat_min"], SEARCH_BBOX["lon_max"]],
    [SEARCH_BBOX["lat_max"], SEARCH_BBOX["lon_max"]],
    [SEARCH_BBOX["lat_max"], SEARCH_BBOX["lon_min"]],
    [SEARCH_BBOX["lat_min"], SEARCH_BBOX["lon_min"]],
]
folium.PolyLine(bbox_coords, color="gray", weight=2, dash_array="6").add_to(m)

print(f"Search grid: {len(lats)}×{len(lons)} = {len(lats)*len(lons)} circles, "
      f"radius {SEARCH_RADIUS_M/1000:.0f} km")
print(f"Found {len(all_pharmacies)} pharmacies (green = active, red = closed)")
m


Search grid: 5×5 = 25 circles, radius 30 km
Found 224 pharmacies (green = active, red = closed)


In [5]:
pharmacies = pd.DataFrame(all_pharmacies)

closed = pharmacies["business_status"] == "CLOSED_PERMANENTLY"
if closed.any():
    print(f"Excluding {closed.sum()} permanently closed pharmacies:")
    for _, row in pharmacies[closed].iterrows():
        print(f"  - {row['name']}, {row['address']}")

pharmacies = pharmacies[~closed].reset_index(drop=True)
print(f"{len(pharmacies)} active pharmacies")
pharmacies[["name", "city", "address", "lat", "lon"]]


224 active pharmacies


,name,city,address,lat,lon
0,Kinney Drugs Pharmacy,Bomoseen,"34 Rte 30 N, Bomoseen, VT 05732, USA",43.609006,-73.207854
1,Walgreens Pharmacy,Middlebury,"263 Court St, Middlebury, VT 05753, USA",44.003496,-73.153925
2,Porter Hospital,Middlebury,"115 Porter Dr, Middlebury, VT 05753, USA",43.999727,-73.168439
3,Walmart Pharmacy,Ticonderoga,"1134 Wicker St, Ticonderoga, NY 12883, USA",43.856698,-73.433198
4,CVS Pharmacy,Whitehall,"170 Broadway SUITE 1, Whitehall, NY 12887, USA",43.547783,-73.406510
...,...,...,...,...,...
219,Familiprix Clinique,Waterville,"347 Rue Gosselin, Waterville, QC J0B 3H0, Canada",45.280265,-71.897881
220,Proxim Chantal Dionne et Éric Portelance,Compton,"6630 Rte Louis-S.-Saint-Laurent, Compton, QC J...",45.246072,-71.828590
221,shoppers drug mart,Magog,"415 Rue Sherbrooke, Magog, QC J1X 2S4, Canada",45.268729,-72.143576
222,Proximed pharmacie affiliée - Amnay Yassine,Magog,"231 Rue Dollard, Magog, QC J1X 0G7, Canada",45.268693,-72.146567


In [6]:
PHARMACIES_ACTIVE = DATA_DIR / "pharmacies_active.json"
PHARMACIES_ACTIVE.write_text(json.dumps(pharmacies.to_dict("records"), indent=2))
print(f"Wrote {len(pharmacies)} active pharmacies to {PHARMACIES_ACTIVE.name}")
print(f"{len(pharmacies)} pharmacies ready for 01-pharmacy_access.ipynb")


Wrote 224 active pharmacies to pharmacies_active.json
224 pharmacies ready for 01-pharmacy_access.ipynb
